# Generating synthetic weather
**Introduction to Climate Modeling · revised November 19, 2026**

In Lab 0c, we separated New York City's seasonal cycle from its day-to-day weather. A smooth seasonal curve tells us that July is usually warmer than January. How could we also generate the warm and cold episodes that occur around that curve?

We will learn a simple probability model from the Central Park observations, then use random numbers to generate new daily temperature records. First we draw each day's weather independently. Then we add memory of the previous day and ask whether that improves the generated weather.

The aim is to reproduce **weather statistics**: the temperature distribution, persistence, and lengths of hot spells. Each generated record is a new possible sequence under the fitted climate; its hot days need not occur on the dates of observed hot days.

All code is supplied. Run the cells in order, make the requested predictions, and use the figures to answer the four **Lab questions**. Allow about **30–40 minutes**, with the final persistence experiment optional.

Download this notebook and open it in the course [LEAP Jupyter environment](https://leap.2i2c.cloud/). The data-reading cell retrieves a fixed copy of the Lab 0c data from our course GitHub and keeps it for later runs. An optional [notebook and data package](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/generative_weather.zip) is available for offline use.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1. Reuse the NYC observations

Use the daily Central Park station temperatures from Lab 0c. The temperature is the average of each day's maximum and minimum, $(T_{max}+T_{min})/2$, in °C. The file also contains ERA5 values; here we use only the observed column, `t_obs`.

- **Training period, 1995–2009:** estimate the seasonal mean, weather variability, persistence, and hot-day threshold.
- **Held-out period, 2010–2014:** compare the generated statistics with observations that were not used in fitting.

We generate weather for the later calendar dates so that the seasons and record lengths match. We do not initialize the generator from the observed weather on those dates. This is a test of how well a climate learned from the earlier period represents later weather statistics.

Run the cell and locate the split between the training and held-out observations.

In [ ]:
data_file = Path("data/centralpark_obs_era5_1995-2014.csv")
SOURCE_URL = 'https://raw.githubusercontent.com/dhruvbalwada/intro-climate-modeling-fall2026/a44a0f875c055ec6cde6d0c48b970687428040df/labs/data/centralpark_obs_era5_1995-2014.csv'
SOURCE_SHA256 = 'dc320f02b00dd82dcb7150683fe29fcfe5fe7d7d22477b85d6965819588a2d4e'

if not data_file.exists():
    print("Downloading the fixed classroom data file...")
    with urlopen(SOURCE_URL, timeout=45) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != SOURCE_SHA256:
        raise ValueError("Data do not match the class version. Download the optional notebook-and-data package linked above.")
    data_file.parent.mkdir(parents=True, exist_ok=True)
    data_file.write_bytes(raw)

frame = pd.read_csv(data_file, index_col=0, parse_dates=True)
observed = frame["t_obs"].sort_index()
expected_dates = pd.date_range("1995-01-01", "2014-12-31", freq="D")
if not observed.index.equals(expected_dates) or observed.isna().any():
    raise ValueError("This exercise expects the supplied complete daily observation record.")

train = observed.loc[:"2009-12-31"]
test = observed.loc["2010-01-01":]
dates = test.index
print(f"Training: {len(train)} days; held out: {len(test)} days")

fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
ax.plot(train.index, train, color="#777777", lw=0.6, label="Training observations")
ax.plot(test.index, test, color="#111111", lw=0.6, label="Held-out observations")
ax.axvline(dates[0], color="#c54b26", ls="--", lw=1)
ax.set(ylabel="Daily temperature (°C)", title="The NYC temperature record from Lab 0c")
ax.legend(loc="upper left")
plt.show()

## 2. Learn the seasonal mean and weather variability

Start with the predictable part of the record: the annual cycle. As in the Keeling curve fits, use sine and cosine terms:

$$\mu(d)=b_0+b_1\sin(2\pi q(d))+b_2\cos(2\pi q(d)).$$

Here $d$ labels a day, and $q(d)$ is the fraction of the calendar year elapsed at the middle of that day. The constant $b_0$ sets the mean level; the sine and cosine together allow the amplitude and timing of the seasonal cycle to be fitted. All three coefficients are learned by least squares using the training years.

Subtract this seasonal mean from each training temperature. The remainder, $T_d-\mu(d)$, is a **weather anomaly**: how much warmer or colder the day was than the fitted seasonal expectation. Estimate the standard deviation of these anomalies in each month, then interpolate between months to obtain $\sigma(d)$. This allows weather variability to differ between summer and winter.

| Quantity | Meaning | Units |
| :--- | :--- | :--- |
| $T_d$ | Daily temperature | °C |
| $\mu(d)$ | Fitted seasonal mean temperature | °C |
| $\sigma(d)$ | Typical size of weather anomalies, measured by their standard deviation | °C |
| $z_d=[T_d-\mu(d)]/\sigma(d)$ | Standardized weather anomaly | Dimensionless |

A value $z_d=1$ means one weather standard deviation warmer than the fitted seasonal mean. For example, if $\mu=25$°C and $\sigma=3$°C, then $z=1$ gives 28°C and $z=-1$ gives 22°C.

Inspect the curve and shaded band below. The band shows **weather variability around the mean**; it is not an uncertainty interval for the fitted mean.

In [ ]:
def year_fraction(index):
    return (index.dayofyear.to_numpy() - 0.5) / (365 + index.is_leap_year.astype(int))

def seasonal_features(index):
    phase = 2*np.pi*year_fraction(index)
    return np.column_stack((np.ones(len(index)), np.sin(phase), np.cos(phase)))

# All fitted quantities below use training observations only.
mean_coefficients = np.linalg.lstsq(seasonal_features(train.index), train.to_numpy(), rcond=None)[0]

def mean_cycle(index):
    return seasonal_features(index) @ mean_coefficients

train_residual = train - mean_cycle(train.index)
monthly_sd = train_residual.groupby(train_residual.index.month).std().reindex(range(1, 13)).to_numpy()

def weather_sd(index):
    month_centres = (np.arange(12) + 0.5) / 12
    return np.interp(year_fraction(index), month_centres, monthly_sd, period=1.0)

mu = mean_cycle(dates)
sigma = weather_sd(dates)
obs_z = (test.to_numpy() - mu) / sigma

calendar = pd.date_range("2011-01-01", "2011-12-31")
phase_days = 365*year_fraction(train.index)
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
ax.scatter(phase_days, train, s=3, color="#888888", alpha=0.12, label="Training observations")
mid = mean_cycle(calendar)
spread = weather_sd(calendar)
ax.plot(np.arange(365)+0.5, mid, color="#111111", lw=2, label="Learned seasonal mean")
ax.fill_between(np.arange(365)+0.5, mid-spread, mid+spread, color="#2374a5", alpha=0.3,
                label="Mean ± one modeled weather standard deviation")
ax.set(xlabel="Position in the year (days)", ylabel="Daily temperature (°C)", xlim=(0, 365))
ax.legend(fontsize=9, loc="upper left")
plt.show()

<a id="response-mean"></a>

**Lab questions**

What would be missing if we generated a year using only the learned seasonal mean? Use the observations and shaded band in the figure.

**Answer:** [Your answer here.]

## 3. Generate independent daily weather

We now need a rule for choosing the anomalies. Assume that standardized anomalies follow a **standard normal distribution**, $N(0,1)$: a bell-shaped distribution with mean zero and standard deviation one. Small anomalies are common; large positive and negative anomalies are less common. This distributional shape is a modeling assumption. The observations set the seasonal mean and spread.

For each day, draw a new random number and convert it to temperature:

$$z_d\sim N(0,1),\qquad T_d=\mu(d)+\sigma(d)z_d.$$

The symbol $\sim$ means “is drawn from.” The second equation scales the random anomaly into °C and adds it to the seasonal mean. Drawing every day independently means that a warm anomaly today gives no information about tomorrow's anomaly.

Generate **200 ensemble members**, each a complete five-year record. Every member uses the same fitted $\mu$ and $\sigma$, but different random draws. More members let us estimate the generator's statistics more precisely; they do not provide more observed years for fitting the model.

The supplied function has a persistence setting, `phi`. Start with `phi=0`, which gives independent anomalies. We will explain the other setting in Section 4. The fixed random seed makes these figures repeatable. To obtain a different ensemble, change `SEED` and rerun this cell and the cells below it.

**Predict before running:** Should the generated temperature match the observation on a particular day? What broad feature of the summer record should it share?

In [ ]:
N_MEMBERS = 200
SEED = 12

def generate_weather(phi, n_members=N_MEMBERS, seed=SEED):
    if not -1 < phi < 1:
        raise ValueError("Use -1 < phi < 1 for this stationary generator.")
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal((n_members, len(dates)))
    z = np.empty_like(noise)
    z[:, 0] = noise[:, 0]  # A draw from the stationary N(0, 1) distribution.
    for day in range(1, len(dates)):
        z[:, day] = phi*z[:, day-1] + np.sqrt(1-phi**2)*noise[:, day]
    temperature = mu[None, :] + sigma[None, :]*z
    return temperature, z

iid_temperature, iid_z = generate_weather(phi=0)

summer = (dates >= "2011-06-01") & (dates <= "2011-08-31")
fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
ax.plot(dates[summer], test.to_numpy()[summer], color="#111111", lw=1.3, label="Observed weather")
ax.plot(dates[summer], iid_temperature[0, summer], color="#2374a5", lw=1.1,
        label="One independent generated sequence")
ax.plot(dates[summer], mu[summer], color="#777777", ls="--", label="Learned seasonal mean")
ax.set(ylabel="Daily temperature (°C)", title="A new realization of summer weather")
ax.legend(fontsize=9)
plt.show()

<a id="response-persistence"></a>

**Lab questions**

Inspect the summer record. Do independent daily draws seem likely to reproduce the lengths of warm and cold episodes? Predict what would change if tomorrow retained some of today's anomaly.

**Answer:** [Your answer here.]

## 4. Add memory of the previous day

Warm and cold weather often lasts several days. Standardize the training anomalies, $z_d=[T_d-\mu(d)]/\sigma(d)$, then fit a straight line relating each day's anomaly to the previous day's anomaly. We fit through zero because these anomalies are defined relative to the seasonal mean. The fitted slope, $\phi$, measures how much of yesterday's standardized anomaly is carried into today.

Use that fitted relationship as a rule for generating new anomalies:

$$z_d=\underbrace{\phi z_{d-1}}_{\text{memory of yesterday}}+
\underbrace{\sqrt{1-\phi^2}\,\epsilon_d}_{\text{new random contribution}},
\qquad \epsilon_d\sim N(0,1).$$

Each $\epsilon_d$ is a fresh, independent standard normal draw. For positive $\phi$, a warm anomaly makes another warm anomaly more likely the next day. It does not guarantee one: the new random contribution can change the sign. This is a **first-order autoregressive model**, or **AR(1)**; it uses one previous day explicitly.

Why reduce the random contribution by $\sqrt{1-\phi^2}$? The memory term already contributes variability. This factor keeps the long-run standard deviation of $z$ equal to one as we change the memory. Both generators therefore have the same normal distribution at any particular calendar day, with the same temperature mean $\mu(d)$ and spread $\sigma(d)$. We are changing how days are connected.

**Connection to the EBM:** In our EBM, a temperature anomaly decays toward equilibrium when additional forcing is removed. Here, if the new random contributions were turned off, an anomaly would decay as $\phi,\phi^2,\phi^3,\ldots$ over successive days. Continued random inputs maintain the weather variability. The shared idea is **relaxation plus new inputs**. This statistical model learns a memory coefficient from temperatures; it does not calculate an energy budget.

Run the cell. The scatter plot shows the fitted relationship in the training observations. Then compare the three summer sequences on the same temperature scale. The two generated sequences use the same underlying random draws, so the change comes from adding memory.

In [ ]:
z_train = train_residual.to_numpy() / weather_sd(train.index)
phi_fit = np.dot(z_train[:-1], z_train[1:]) / np.dot(z_train[:-1], z_train[:-1])
print(f"Learned persistence coefficient phi = {phi_fit:.3f}")

# Reuse the same seed and seasonal distribution as the independent generator.
persistent_temperature, persistent_z = generate_weather(phi=phi_fit)

fig, ax = plt.subplots(figsize=(5.5, 4), layout="constrained")
ax.scatter(z_train[:-1], z_train[1:], s=5, alpha=0.12, color="#777777")
x_line = np.array([-4.0, 4.0])
ax.plot(x_line, phi_fit*x_line, color="#c54b26", lw=2,
        label=f"Fitted slope = {phi_fit:.2f}")
ax.axhline(0, color="#bbbbbb", lw=0.7)
ax.axvline(0, color="#bbbbbb", lw=0.7)
ax.set(xlabel="Yesterday's standardized anomaly", ylabel="Today's standardized anomaly",
       title="Learning memory from the training years", xlim=(-4, 4), ylim=(-4, 4))
ax.legend(fontsize=9)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, sharey=True, layout="constrained")
for ax, values, title, color in zip(
        axes,
        [test.to_numpy(), iid_temperature[0], persistent_temperature[0]],
        ["Observed weather", "Independent generator", "Generator with learned persistence"],
        ["#111111", "#2374a5", "#c54b26"]):
    ax.plot(dates[summer], values[summer], color=color, lw=1.3)
    ax.plot(dates[summer], mu[summer], color="#777777", ls="--", lw=1)
    ax.set(title=title, ylabel="°C")
axes[-1].set_xlabel("Summer 2011 calendar dates")
plt.show()

## 5. Check the distribution and the sequence

A summer plot shows only one member. Now evaluate all 200 members against the five held-out years.

**Left panel — how often do different anomalies occur?** The histogram shows the distribution of standardized anomalies. A realistic generator should reproduce the observed spread and the frequency of unusually warm or cold anomalies. The generated histograms pool all members.

**Right panel — how long does memory last?** Correlate anomalies separated by a given number of days. A lag of one compares today with tomorrow; a lag of five compares today with five days later. Correlation near one means strong similarity, while correlation near zero means little linear association. Each generated curve is the average of the correlations calculated for individual members.

We remove the seasonal cycle and divide by the seasonal spread before calculating these statistics. Otherwise the predictable succession of warm summer days could look like persistence of weather anomalies.

**Predict before running:** Which panel should change most when we add memory? Then compare your prediction with the figures.

In [ ]:
def lag_correlation(values, lag=1):
    values = np.atleast_2d(values)
    a, b = values[:, :-lag], values[:, lag:]
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    return (a*b).sum(axis=1) / np.sqrt((a*a).sum(axis=1)*(b*b).sum(axis=1))

lags = np.arange(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
bins = np.linspace(-4, 4, 41)
for values, label, color in [(obs_z, "Held-out observations", "#111111"),
                              (iid_z, "Independent generator", "#2374a5"),
                              (persistent_z, "Persistent generator", "#c54b26")]:
    axes[0].hist(values.ravel(), bins=bins, density=True, histtype="step", lw=1.8,
                 label=label, color=color)
    acf = [lag_correlation(values, lag).mean() for lag in lags]
    axes[1].plot(lags, acf, marker="o", color=color, label=label)
axes[0].set(xlabel="Standardized weather anomaly", ylabel="Probability density")
axes[1].set(xlabel="Lag (days)", ylabel="Anomaly correlation", xticks=lags, ylim=(-0.1, 1))
axes[1].axhline(0, color="#999999", lw=0.8)
axes[0].legend(fontsize=9)
plt.show()

<a id="response-histogram"></a>

**Lab questions**

Can two generators have very similar histograms but produce different kinds of weather sequences? Use both panels as evidence. Does fitting the relationship between adjacent days also guarantee a match at longer lags?

**Answer:** [Your answer here.]

## 6. Do we generate realistic hot spells?

A useful weather generator should reproduce events that depend on a sequence of days. Call a day **hot** when its temperature exceeds the **95th percentile of all training temperatures**: about 5% of training days exceed this fixed threshold. Because the threshold is in °C and is the same all year, most hot days occur in summer. It is a classroom definition, rather than an official heatwave definition.

For each generated five-year record, calculate two quantities: the **total number of hot days** and the **longest consecutive run of hot days**. The table summarizes the 200 members by their median. The histograms show the spread across whole five-year records; the black line is the single observed five-year value.

**Predict before running:** Both generators have the same temperature distribution on any particular day. Should adding memory change the expected total number of hot days, their clustering into spells, or both? A finite ensemble can still give slightly different medians.

In [ ]:
hot_threshold = float(train.quantile(0.95))
print(f"Hot-day threshold from training data: {hot_threshold:.1f} °C")

def longest_hot_run(values):
    values = np.atleast_2d(values)
    run = np.zeros(values.shape[0], dtype=int)
    longest = np.zeros_like(run)
    for hot_today in (values > hot_threshold).T:
        run = np.where(hot_today, run + 1, 0)
        longest = np.maximum(longest, run)
    return longest

def statistics(temperature, z):
    temperature, z = np.atleast_2d(temperature), np.atleast_2d(z)
    return {
        "Hot days (five years)": (temperature > hot_threshold).sum(axis=1),
        "Longest hot spell (days)": longest_hot_run(temperature),
        "Lag-1 anomaly correlation": lag_correlation(z),
        "Standardized anomaly SD": z.std(axis=1),
    }

observed_stats = statistics(test.to_numpy(), obs_z)
iid_stats = statistics(iid_temperature, iid_z)
persistent_stats = statistics(persistent_temperature, persistent_z)

summary = pd.DataFrame({
    "Held-out observations": {k: v[0] for k, v in observed_stats.items()},
    "Independent generator: median": {k: np.median(v) for k, v in iid_stats.items()},
    "Persistent generator: median": {k: np.median(v) for k, v in persistent_stats.items()},
})
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, key in zip(axes, ["Hot days (five years)", "Longest hot spell (days)"]):
    low = min(iid_stats[key].min(), persistent_stats[key].min(), observed_stats[key][0])
    high = max(iid_stats[key].max(), persistent_stats[key].max(), observed_stats[key][0])
    bin_width = max(1, int(np.ceil((high-low+1)/20)))
    edges = np.arange(low-0.5, high+bin_width+0.5, bin_width)
    for values, color, label in [(iid_stats[key], "#2374a5", "Independent"),
                                  (persistent_stats[key], "#c54b26", "Persistent")]:
        ax.hist(values, bins=edges, alpha=0.45, color=color, label=label)
    ax.axvline(observed_stats[key][0], color="#111111", lw=2, label="Observed")
    ax.set(xlabel=key, ylabel="Number of generated records")
axes[0].legend(fontsize=9)
plt.show()

<a id="response-hot-spells"></a>

**Lab questions**

How does adding persistence change the longest hot spells and the spread of hot-day counts? Compare both with the observed record. Which features improved, and what discrepancy remains?

**Answer:** [Your answer here.]

## Optional experiment: change the memory

Keep the fitted seasonal mean and variability fixed. Change `PHI_EXPERIMENT` below and rerun the cell. Try `0.0`, `phi_fit`, and `0.9`.

**Predict first:** As $\phi$ approaches one, will warm and cold episodes become longer or shorter? Should the overall anomaly spread increase?

The summer plot makes the change in sequencing visible; the table compares statistics across all members. This is a sensitivity experiment, not a new fit to observations. A larger memory coefficient does not necessarily make the generated weather more realistic.

In [ ]:
PHI_EXPERIMENT = 0.9
experiment_temperature, experiment_z = generate_weather(phi=PHI_EXPERIMENT)
experiment_stats = statistics(experiment_temperature, experiment_z)
comparison = pd.DataFrame({
    "Independent: median": {k: np.median(v) for k, v in iid_stats.items()},
    "Learned persistence: median": {k: np.median(v) for k, v in persistent_stats.items()},
    f"Experiment phi={PHI_EXPERIMENT:g}: median": {k: np.median(v) for k, v in experiment_stats.items()},
})
display(comparison.round(3))

fig, axes = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True, sharey=True,
                         layout="constrained")
for ax, values, title, color in zip(
        axes, [persistent_temperature[0], experiment_temperature[0]],
        [f"Learned memory: phi = {phi_fit:.2f}", f"Experiment: phi = {PHI_EXPERIMENT:g}"],
        ["#c54b26", "#6a51a3"]):
    ax.plot(dates[summer], values[summer], color=color, lw=1.3)
    ax.plot(dates[summer], mu[summer], color="#777777", ls="--", lw=1)
    ax.set(title=title, ylabel="Daily temperature (°C)")
axes[-1].set_xlabel("Summer 2011 calendar dates")
plt.show()


## What makes this a generative model?

We learned parameters from observations, specified a probability model, and sampled new weather sequences from it. New random draws produce new realizations while the fitted parameters remain fixed. In the regression notebook we assessed predictions against observed values on individual dates. Here we assess whether generated records reproduce observed distributions, memory, and events. A good match in one of those tests does not ensure a good match in the others.

Our model assumes Gaussian anomalies and a single memory coefficient. It represents the seasonal climate learned from 1995–2009, repeated through later years. It has no changing emissions or external forcing to drive a changing climate. More ensemble members reduce sampling noise in our model statistics; they cannot fix those assumptions or the limited observational record.

### Connection to generative AI

This is a classical statistical weather generator. It makes the idea of learning a distribution and generating samples visible with a few parameters. More flexible generative models can learn relationships across many atmospheric variables and locations.

For example, [GenCast](https://www.nature.com/articles/s41586-024-08252-9) uses a learned conditional diffusion model to sample possible future global weather states, given recent weather states. Repeated sampling produces an ensemble of forecasts. Our generator is not conditioned on the observed initial weather: it samples possible sequences under a fixed seasonal climate. The common idea is to generate realizations from a learned probability model; the information used to generate them and the tests of success differ.

### Sources and modeling choices

- Data and starting point: [Lab 0c — Climate of New York City](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/lab0c_nyc_climate.html). NOAA NCEI GHCN-Daily, Central Park station USW00094728; daily temperature defined as $(T_{max}+T_{min})/2$. The data-reading cell pins the course GitHub version and checksum. The optional offline package also includes `data/provenance.json`.
- Classical background: Richardson (1981), [Stochastic simulation of daily precipitation, temperature, and solar radiation](https://doi.org/10.1029/WR017i001p00182). This lab uses a simplified temperature-only generator with a sinusoidal mean, seasonally varying spread, and one AR(1) coefficient; it does not implement the full Richardson model.
- Modern example: Price et al., [Probabilistic weather forecasting with machine learning](https://doi.org/10.1038/s41586-024-08252-9), published online December 2024, in *Nature* 637 (2025).